In [1]:
from pyspark.sql import SparkSession

from utility import data_preparation,preprocessing_normal

/home/michele/.local/lib/python3.10/site-packages/beir/datasets/data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
# Create SparkSession 
spark = SparkSession.builder.master("local[8]").appName("all-doc-pairs-similarity.com").config("spark.driver.memory", "10g").getOrCreate()

sc = spark.sparkContext

23/05/22 16:21:00 WARN Utils: Your hostname, michele-Veriton-M2631G resolves to a loopback address: 127.0.1.1; using 192.168.1.6 instead (on interface enp3s0)
23/05/22 16:21:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/05/22 16:21:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
corpus, keys = data_preparation("scifact")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [5]:
#vectorized_docs_rdd=preprocessing_spark()
vectorized_docs_rdd=preprocessing_normal(corpus,keys,sc)

documents cleaning:   0%|          | 0/5183 [00:00<?, ?it/s]

In [6]:
vectorized_docs_rdd.collect()

[('4983',
  <1x31637 sparse matrix of type '<class 'numpy.float64'>'
  	with 98 stored elements in Compressed Sparse Row format>),
 ('5836',
  <1x31637 sparse matrix of type '<class 'numpy.float64'>'
  	with 117 stored elements in Compressed Sparse Row format>),
 ('7912',
  <1x31637 sparse matrix of type '<class 'numpy.float64'>'
  	with 63 stored elements in Compressed Sparse Row format>),
 ('18670',
  <1x31637 sparse matrix of type '<class 'numpy.float64'>'
  	with 114 stored elements in Compressed Sparse Row format>),
 ('19238',
  <1x31637 sparse matrix of type '<class 'numpy.float64'>'
  	with 85 stored elements in Compressed Sparse Row format>),
 ('33370',
  <1x31637 sparse matrix of type '<class 'numpy.float64'>'
  	with 120 stored elements in Compressed Sparse Row format>),
 ('36474',
  <1x31637 sparse matrix of type '<class 'numpy.float64'>'
  	with 65 stored elements in Compressed Sparse Row format>),
 ('54440',
  <1x31637 sparse matrix of type '<class 'numpy.float64'>'
  	wit

In [7]:
def Map(doc_pair):
    
    id, doc = doc_pair
    
    non_zero_terms=doc.nonzero()
    
    term__doc_ids=[]
    
    for term_index in non_zero_terms[1]:
        if doc.argmax()==term_index:
            term__doc_ids.append((term_index,id))
    
    return term__doc_ids

term_listDocIds_pairs_rdd=vectorized_docs_rdd.flatMap(Map).groupByKey()

In [8]:
vectorized_docs_broadcast=sc.broadcast(dict(vectorized_docs_rdd.collect()))

In [9]:
import itertools

threshold=sc.broadcast(0.8)

def Reduce(id_list):
    
    similarities=[]
    
    for id1, id2 in itertools.combinations(id_list,2):

        similarity=vectorized_docs_broadcast.value[id1].dot(vectorized_docs_broadcast.value[id2].transpose())
        
        if similarity[0,0]>=threshold.value:
            similarities.append(((id1,id2),similarity[0,0]))
        
    return similarities

similar_doc_pairs=term_listDocIds_pairs_rdd.values().flatMap(Reduce).persist()

In [10]:
similar_doc_pairs.collect()

[(('2452989', '39506601'), 0.8090001821068815),
 (('8909176', '32712381'), 0.8479062050792276),
 (('36233757', '37768883'), 0.9023826206284256),
 (('13023410', '21793890'), 0.8686206018987505),
 (('1831916', '24625323'), 0.8198176877640402),
 (('8963413', '52180874'), 0.8706614469733832)]

In [14]:
dict(zip(keys,corpus))['2452989']

'A global role for KLF1 in erythropoiesis revealed by ChIP-seq in primary erythroid cells. KLF1 regulates a diverse suite of genes to direct erythroid cell differentiation from bipotent progenitors. To determine the local cis-regulatory contexts and transcription factor networks in which KLF1 operates, we performed KLF1 ChIP-seq in the mouse. We found at least 945 sites in the genome of E14.5 fetal liver erythroid cells which are occupied by endogenous KLF1. Many of these recovered sites reside in erythroid gene promoters such as Hbb-b1, but the majority are distant to any known gene. Our data suggests KLF1 directly regulates most aspects of terminal erythroid differentiation including production of alpha- and beta-globin protein chains, heme biosynthesis, coordination of proliferation and anti-apoptotic pathways, and construction of the red cell membrane and cytoskeleton by functioning primarily as a transcriptional activator. Additionally, we suggest new mechanisms for KLF1 cooperati

In [15]:
dict(zip(keys,corpus))['39506601']

'Novel roles for KLF1 in erythropoiesis revealed by mRNA-seq. KLF1 (formerly known as EKLF) regulates the development of erythroid cells from bi-potent progenitor cells via the transcriptional activation of a diverse set of genes. Mice lacking Klf1 die in utero prior to E15 from severe anemia due to the inadequate expression of genes controlling hemoglobin production, cell membrane and cytoskeletal integrity, and the cell cycle. We have recently described the full repertoire of KLF1 binding sites in vivo by performing KLF1 ChIP-seq in primary erythroid tissue (E14.5 fetal liver). Here we describe the KLF1-dependent erythroid transcriptome by comparing mRNA-seq from Klf1(+/+) and Klf1(-/-) erythroid tissue. This has revealed novel target genes not previously obtainable by traditional microarray technology, and provided novel insights into the function of KLF1 as a transcriptional activator. We define a cis-regulatory module bound by KLF1, GATA1, TAL1, and EP300 that coordinates a core s

In [16]:
#spark.stop()